In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = pd.Series(iris.target)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

default_pipeline = Pipeline(
    steps=[
        ('model', DecisionTreeClassifier(random_state=42))
    ]
)

default_pipeline.fit(X_train, y_train)

y_pred_default = default_pipeline.predict(X_test)
print("--- Default Model Performance ---")
print(f"Test Accuracy: {accuracy_score(y_test, y_pred_default):.4f}")
print(classification_report(y_test, y_pred_default, target_names=iris.target_names))

--- Default Model Performance ---
Test Accuracy: 0.9333
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.90      0.90      0.90        10
   virginica       0.90      0.90      0.90        10

    accuracy                           0.93        30
   macro avg       0.93      0.93      0.93        30
weighted avg       0.93      0.93      0.93        30



In [3]:
from sklearn.model_selection import GridSearchCV

tuning_pipeline = Pipeline(
    steps=[
        ('model', DecisionTreeClassifier(random_state=42))
    ]
)

param_grid = {
    'model__criterion': ['gini', 'entropy'], 
    'model__max_depth': [3, 4, 5, None],     
    'model__min_samples_split': [2, 5, 10],  
    'model__min_samples_leaf': [1, 2, 4]     
}

grid_search = GridSearchCV(
    estimator=tuning_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("--- GridSearchCV Tuning Results (No Scaling) ---")
print("Best Parameters Found:", grid_search.best_params_)
print(f"Best CV Accuracy: {grid_search.best_score_:.4f}")

best_model_pipeline = grid_search.best_estimator_
y_pred_tuned = best_model_pipeline.predict(X_test)

print("\n--- Tuned Model Performance on Test Set ---")
print(f"Test Accuracy: {accuracy_score(y_test, y_pred_tuned):.4f}")
print(classification_report(y_test, y_pred_tuned, target_names=iris.target_names))

importances = best_model_pipeline.named_steps['model'].feature_importances_
for name, importance in zip(iris.feature_names, importances):
    print(f"{name:20s}: {importance:.4f}")

--- GridSearchCV Tuning Results (No Scaling) ---
Best Parameters Found: {'model__criterion': 'gini', 'model__max_depth': 4, 'model__min_samples_leaf': 1, 'model__min_samples_split': 2}
Best CV Accuracy: 0.9417

--- Tuned Model Performance on Test Set ---
Test Accuracy: 0.9333
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.90      0.90      0.90        10
   virginica       0.90      0.90      0.90        10

    accuracy                           0.93        30
   macro avg       0.93      0.93      0.93        30
weighted avg       0.93      0.93      0.93        30

sepal length (cm)   : 0.0063
sepal width (cm)    : 0.0169
petal length (cm)   : 0.5656
petal width (cm)    : 0.4112
